# CS383: Data Science and Machine Learning
## Lecture 4 Exercises — Datetime Handling, Data Cleaning, Reshaping

**Make a copy of this notebook before you start** (do not edit this original). Fill in every `__________` blank, then run all cells top to bottom before submitting. This notebook is graded with Otter Grader — do not delete or modify the setup cells.

### Setup

Run this first — it rebuilds the NYC 311 dataset and applies the same datetime parsing, resolution-time calculation, duplicate removal, and borough text cleanup covered in Lecture 4, so you're starting from the same clean state the lab and challenge assume.

In [ ]:
import sqlite3
import numpy as np
import pandas as pd
import requests

SOCRATA_URL = "https://data.cityofnewyork.us/resource/erm2-nwe9.json"

try:
    response = requests.get(
        SOCRATA_URL,
        params={
            "$limit": 8000,
            "$order": "created_date DESC",
            "$select": "unique_key,complaint_type,borough,created_date,closed_date",
        },
        timeout=8,
    )
    response.raise_for_status()
    complaints_df = pd.DataFrame(response.json())
    live = True
except Exception:
    # Offline fallback, in case there is no internet connection in the room.
    rng = np.random.default_rng(383)
    n = 8000
    complaint_types = ["Noise - Residential", "Illegal Parking", "HEAT/HOT WATER",
                        "Blocked Driveway", "Street Condition", "Water System",
                        "PAINT/PLASTER", "Damaged Tree", "Sewer", "Rodent"]
    boroughs_list = ["MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"]

    created = pd.date_range("2026-01-01", periods=n, freq="min")
    still_open = rng.random(n) < 0.15  # about 15% of complaints have no closed_date yet
    resolution_hours = rng.gamma(shape=2.0, scale=20.0, size=n)
    closed = created + pd.to_timedelta(resolution_hours, unit="h")
    closed_str = np.where(still_open, None, closed.astype(str))

    complaints_df = pd.DataFrame({
        "unique_key": np.arange(1, n + 1).astype(str),  # Socrata's API returns everything as text
        "complaint_type": rng.choice(
            complaint_types, size=n,
            p=[0.18, 0.15, 0.14, 0.10, 0.10, 0.09, 0.08, 0.06, 0.05, 0.05],
        ),
        "borough": rng.choice(boroughs_list, size=n, p=[0.22, 0.32, 0.26, 0.16, 0.04]),
        "created_date": created.astype(str),
        "closed_date": closed_str,
    })
    live = False

# Deliberately introduce some real-world messiness, so there's something to clean
# in Part 2 regardless of whether today's data is live or offline.
rng2 = np.random.default_rng(99)
messy_idx = rng2.choice(complaints_df.index, size=15, replace=False)
complaints_df.loc[messy_idx, "borough"] = complaints_df.loc[messy_idx, "borough"].str.lower()
complaints_df = pd.concat([complaints_df, complaints_df.sample(6, random_state=1)], ignore_index=True)

print(f"{'Live' if live else 'Offline fallback'} data: {len(complaints_df):,} 311 records")
complaints_df.head()

In [ ]:
print("Before parsing:")
print(complaints_df[["created_date", "closed_date"]].dtypes)

complaints_df["created_date"] = pd.to_datetime(complaints_df["created_date"])
complaints_df["closed_date"] = pd.to_datetime(complaints_df["closed_date"])

print("\nAfter parsing:")
print(complaints_df[["created_date", "closed_date"]].dtypes)

In [ ]:
complaints_df["resolution_time_hours"] = (
    complaints_df["closed_date"] - complaints_df["created_date"]
).dt.total_seconds() / 3600

complaints_df[["created_date", "closed_date", "resolution_time_hours"]].head(10)

In [ ]:
print(f"Duplicate rows: {complaints_df.duplicated().sum()}")

complaints_df = complaints_df.drop_duplicates().reset_index(drop=True)
print(f"Rows after dropping duplicates: {len(complaints_df):,}")

In [ ]:
complaints_df["borough"] = complaints_df["borough"].str.strip().str.upper()
print(complaints_df["borough"].unique())

---

## Exercise 1 — Put It Together

Fill in the blanks below.

**1. Which month name had the most complaints created?**

In [ ]:
complaints_df["created_month_name"] = complaints_df["created_date"].dt.__________()
complaints_df["created_month_name"].value_counts().head(1)

**2. Confirm there are no duplicate rows left, then check for any remaining missing `borough` values.**

In [ ]:
print(f"Duplicate rows remaining: {complaints_df.__________().sum()}")
print(f"Missing borough values: {complaints_df['borough'].__________().sum()}")

---

## Exercise 2 — Reflection (Exit Ticket)

Answer the following in your own words.

1. Why did some rows end up with `NaT` in `closed_date` after parsing, and why shouldn't you just fill those in with an average resolution time?
2. What's the difference between `pd.concat()` and `.merge()`?
3. Give one situation where you'd want a table in wide format, and one where you'd want it in long format.
4. `unique_key` came in as text instead of a number — why might that happen with real API data, and how did we fix it?
5. What question do you still have about datetime handling or data cleaning before Assignment 1?

**Your responses:**

1.  
2.  
3.  
4.  
5.  

## Optional Challenge

### Step 1 — Exclude the still-open complaints

In [ ]:
# Build a version of complaints_df containing only complaints that have actually been closed
# Your code here


### Step 2 — Find the hour of day (0-23) with the highest average `resolution_time_hours`

In [ ]:
# Your code here


### Step 3 — Reshape with `pivot_table`

Hour of day as the index, average resolution time as the value.

In [ ]:
# Your code here


### Step 4 — Reflect

Does the pattern you found surprise you? Explain in one or two sentences.

**Your answer:**

